# 19A — V5 DEV Source Verification Backfill Gate

**Purpose:** source-QA only.

This notebook assembles the repaired Batch 1–8 DEV event corpus and checks whether every event row has traceable source support before the final event freeze.

It **must not**:

- alter roster membership,
- delete or relabel an event,
- change polarity/year/axis,
- inspect pairability or chronology balance,
- generate astrology,
- inspect Control,
- research CONFIRM.

Any eligible row that does not automatically pass source traceability is emitted to a manual-review worklist.  
**Final event freeze remains blocked until every eligible row is source-verified.**


In [2]:

from pathlib import Path
from datetime import datetime
import hashlib, json, re, time, unicodedata
import pandas as pd
import numpy as np

NOTEBOOK_VERSION="SAJU_ML_V5_SOURCE_VERIFICATION_BACKFILL_20260817"

def find_repo_root(start=None):
    p=Path(start or Path.cwd()).resolve()
    for c in [p]+list(p.parents):
        if (c/"saju_engine.py").exists():
            return c
    raise FileNotFoundError("Run inside Chartpalja repo.")

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

def norm(s):
    s=unicodedata.normalize("NFKD",str(s))
    s="".join(c for c in s if not unicodedata.combining(c))
    s=s.casefold()
    s=re.sub(r"\s+"," ",s)
    return s.strip()

ROOT=find_repo_root()
BASE=ROOT/"research/ml/artifacts/v5_event_collection_repaired"
POST=BASE/"results_post_repair"
RES=BASE/"results"
OUT=ROOT/"research/ml/artifacts/v5_source_verification"
CORPUS=ROOT/"research/ml_corpus/v5_ground_truth"
OUT.mkdir(parents=True,exist_ok=True)

PROTO=CORPUS/"V5_SOURCE_VERIFICATION_BACKFILL_PROTOCOL.json"
if not PROTO.exists(): raise FileNotFoundError(PROTO)
protocol=json.load(open(PROTO,encoding="utf-8"))
assert protocol["status"]=="PREDECLARED_BEFORE_CORPUS_SOURCE_QA"

paths=[]
for b in [1,2,3]:
    paths.append((
        b,
        POST/f"batch_{b:02d}"/f"V5_DEV_EVENT_BATCH_{b:02d}_EVENTS_POST_REPAIR.csv"
    ))
for b in [4,5,6,7,8]:
    paths.append((
        b,
        RES/f"batch_{b:02d}"/f"V5_DEV_EVENT_BATCH_{b:02d}_EVENTS.csv"
    ))

for b,p in paths:
    if not p.exists():
        raise FileNotFoundError(f"Missing Batch {b} event file: {p}")

frames=[]
for b,p in paths:
    x=pd.read_csv(p)
    x["source_batch_file"]=str(p.relative_to(ROOT))
    x["event_row_in_batch"]=np.arange(1,len(x)+1)
    frames.append(x)

events=pd.concat(frames,ignore_index=True,sort=False)

# Stable event row id, independent of row ordering.
def row_id(r):
    raw="|".join([
        str(r.get("subject_id","")),
        str(r.get("event_year","")),
        str(r.get("polarity","")),
        str(r.get("event_type","")),
        str(r.get("event_description","")),
        str(r.get("source_url",""))
    ])
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:20]

events["event_row_id"]=events.apply(row_id,axis=1)

assert events.event_row_id.is_unique, "Duplicate event_row_id detected."
assert events.subject_id.nunique()==160, events.subject_id.nunique()
assert set(events.batch_id.astype(int))==set(range(1,9))

assembled_path=OUT/"V5_DEV_EVENTS_REPAIRED_ASSEMBLED_PRE_SOURCE_QA.csv"
events.to_csv(assembled_path,index=False)

print("ASSEMBLY PREFLIGHT PASS")
print("subjects:",events.subject_id.nunique())
print("event rows:",len(events))
print("eligible rows:",int((~events.exclude.astype(bool)).sum()))
print("excluded audit rows:",int(events.exclude.astype(bool).sum()))


ASSEMBLY PREFLIGHT PASS
subjects: 160
event rows: 481
eligible rows: 405
excluded audit rows: 76


## 1. Build immutable source-verification worklist

In [3]:

url_re=re.compile(r"^https?://",re.I)

keyword_families={
    "title":["title","champion","championship","gold","medal","winner","won","final","cup"],
    "election":["election","elected","electoral","vote","mayor","president","prime minister","minister"],
    "office":["appointed","appointment","chair","chairman","ceo","minister","president","director","justice","judge","leader"],
    "resign":["resign","resignation","dismiss","removed","lost","defeat","ended","fired"],
    "award":["award","prize","cesar","césar","oscar","academy","grammy","cannes","venice","berlin","palme","lion","bear"],
    "release":["film","album","song","book","series","recording","project","released","premiere","publication"],
}

def event_keyword_family(event_type):
    et=norm(event_type)
    if any(k in et for k in ["title","olympic","championship","final","medal","ranking","cup","tournament","grand_slam"]):
        return "title"
    if any(k in et for k in ["election","mayor"]):
        return "election"
    if any(k in et for k in ["accession","chair","ceo","minister","judge","justice","leadership","founder","president"]):
        return "office"
    if any(k in et for k in ["loss","resign","dismiss","removal","collapse","exit"]):
        return "resign"
    if any(k in et for k in ["award","prize","grammy","cesar","oscar","cannes","venice","berlin"]):
        return "award"
    if any(k in et for k in ["film","album","song","record","project","publication","release","breakthrough"]):
        return "release"
    return ""

work=events[[
    "event_row_id","batch_id","subject_id","name","preassigned_axis",
    "event_year","polarity","event_type","event_description",
    "source_url","source_title","source_publisher","source_date",
    "source_quality","exclude","exclude_reason","source_batch_file"
]].copy()

work["source_url_valid"]=work.source_url.fillna("").map(lambda x:bool(url_re.match(str(x).strip())))
work["event_keyword_family"]=work.event_type.map(event_keyword_family)

work_path=OUT/"V5_SOURCE_VERIFICATION_WORKLIST.csv"
work.to_csv(work_path,index=False)

print("worklist rows:",len(work))
print("unique source URLs:",work.source_url.nunique())


worklist rows: 481
unique source URLs: 313


## 2. Fetch sources and run conservative automatic traceability checks

In [4]:

import requests
from bs4 import BeautifulSoup

CACHE=OUT/"http_cache"
CACHE.mkdir(parents=True,exist_ok=True)

session=requests.Session()
session.headers.update({
    "User-Agent":"Chartpalja-Saju-Research/1.0 source-verification-audit",
    "Accept-Language":"en-US,en;q=0.8"
})

def cache_key(url):
    return hashlib.sha256(str(url).encode("utf-8")).hexdigest()

def fetch(url,timeout=25):
    key=cache_key(url)
    meta_p=CACHE/f"{key}.json"
    text_p=CACHE/f"{key}.txt"
    if meta_p.exists() and text_p.exists():
        return json.load(open(meta_p,encoding="utf-8")), text_p.read_text(encoding="utf-8",errors="ignore")

    meta={
        "url":url,"ok":False,"status_code":None,"final_url":"",
        "content_type":"","title":"","error":""
    }
    text=""
    try:
        r=session.get(url,timeout=timeout,allow_redirects=True)
        meta["status_code"]=r.status_code
        meta["final_url"]=r.url
        meta["content_type"]=r.headers.get("content-type","")
        if r.status_code < 400:
            html=r.text
            soup=BeautifulSoup(html,"html.parser")
            meta["title"]=soup.title.get_text(" ",strip=True) if soup.title else ""
            for tag in soup(["script","style","noscript","svg"]):
                tag.decompose()
            text=soup.get_text(" ",strip=True)
            text=re.sub(r"\s+"," ",text)
            meta["ok"]=len(text)>=80
        else:
            meta["error"]=f"HTTP_{r.status_code}"
    except Exception as e:
        meta["error"]=repr(e)[:500]

    json.dump(meta,open(meta_p,"w",encoding="utf-8"),ensure_ascii=False,indent=2)
    text_p.write_text(text,encoding="utf-8")
    return meta,text

urls=sorted(set(work.loc[work.source_url_valid,"source_url"].astype(str)))
fetched={}
for i,url in enumerate(urls,1):
    meta,text=fetch(url)
    fetched[url]=(meta,text)
    if i%20==0 or i==len(urls):
        print(f"fetched {i}/{len(urls)}")
    time.sleep(0.15)


fetched 20/313
fetched 40/313
fetched 60/313
fetched 80/313
fetched 100/313
fetched 120/313
fetched 140/313
fetched 160/313
fetched 180/313
fetched 200/313
fetched 220/313
fetched 240/313
fetched 260/313
fetched 280/313
fetched 300/313
fetched 313/313


## 3. Classify rows without changing any event data

In [5]:

def name_tokens(name):
    toks=[x for x in re.findall(r"[a-z0-9]+",norm(name)) if len(x)>=3]
    # Require family/last token when possible, otherwise any informative token.
    return toks[-1:] if toks else []

def row_check(r):
    if not r.source_url_valid:
        return {
            "verification_status":"INVALID_SOURCE_FIELD",
            "fetch_ok":False,"http_status":None,"page_title":"",
            "name_signal":False,"year_signal":False,"keyword_signal":False,
            "review_reason":"Missing/malformed http(s) source_url"
        }

    meta,text=fetched.get(str(r.source_url),({"ok":False},""))
    hay=norm((meta.get("title","") or "")+" "+text[:250000])
    nt=name_tokens(r["name"])
    name_signal=bool(nt) and all(t in hay for t in nt)

    year=str(int(r.event_year)) if pd.notna(r.event_year) else ""
    # Some authoritative bios summarize periods/ranges rather than repeat a precise year.
    year_signal=bool(year and year in hay)

    fam=str(r.event_keyword_family or "")
    kws=keyword_families.get(fam,[])
    keyword_signal=True if not kws else any(norm(k) in hay for k in kws)

    fetch_ok=bool(meta.get("ok",False))
    # Conservative auto-pass:
    # - retrievable page
    # - subject identity visible
    # - at least one of year or event-family signal visible
    auto=fetch_ok and name_signal and (year_signal or keyword_signal)

    if auto:
        st="AUTO_PASS"
        reason=""
    else:
        st="MANUAL_REVIEW"
        reasons=[]
        if not fetch_ok: reasons.append("source_not_retrievable_or_text_too_short")
        if not name_signal: reasons.append("subject_name_not_detected")
        if not year_signal: reasons.append("event_year_not_detected")
        if not keyword_signal: reasons.append("event_keyword_family_not_detected")
        reason=";".join(reasons)

    return {
        "verification_status":st,
        "fetch_ok":fetch_ok,
        "http_status":meta.get("status_code"),
        "page_title":meta.get("title",""),
        "name_signal":name_signal,
        "year_signal":year_signal,
        "keyword_signal":keyword_signal,
        "review_reason":reason
    }

checks=pd.DataFrame([row_check(r) for _,r in work.iterrows()])
qa=pd.concat([work.reset_index(drop=True),checks],axis=1)

qa_path=OUT/"V5_SOURCE_VERIFICATION_RESULTS_AUTO.csv"
qa.to_csv(qa_path,index=False)

manual=qa[
    (qa.verification_status!="AUTO_PASS")
].copy()

manual_path=OUT/"V5_SOURCE_VERIFICATION_MANUAL_REVIEW.csv"
manual.to_csv(manual_path,index=False)

eligible_manual=manual[~manual.exclude.astype(bool)].copy()
eligible_manual_path=OUT/"V5_SOURCE_VERIFICATION_ELIGIBLE_MANUAL_REVIEW.csv"
eligible_manual.to_csv(eligible_manual_path,index=False)

print(qa.verification_status.value_counts(dropna=False))
print()
print("eligible rows requiring manual review:",len(eligible_manual))
print("unique URLs among eligible manual-review rows:",eligible_manual.source_url.nunique())


AUTO_PASS        348
MANUAL_REVIEW    133
Name: verification_status, dtype: int64

eligible rows requiring manual review: 118
unique URLs among eligible manual-review rows: 77


## 4. Freeze the source-QA gate decision

In [6]:

eligible=qa[~qa.exclude.astype(bool)].copy()
eligible_nonpass=eligible[eligible.verification_status!="AUTO_PASS"]

if len(eligible_nonpass)==0:
    status="V5_SOURCE_VERIFICATION_AUTO_GATE_PASS_READY_FOR_FINAL_EVENT_QA"
else:
    status="V5_SOURCE_VERIFICATION_MANUAL_BACKFILL_REQUIRED"

summary={
    "version":"V5_SOURCE_VERIFICATION_GATE_SUMMARY_V1",
    "created_at":datetime.now().isoformat(timespec="seconds"),
    "status":status,
    "subjects_n":int(events.subject_id.nunique()),
    "event_rows_n":int(len(events)),
    "eligible_rows_n":int((~events.exclude.astype(bool)).sum()),
    "excluded_rows_n":int(events.exclude.astype(bool).sum()),
    "unique_source_urls_n":int(work.source_url.nunique()),
    "auto_pass_rows_n":int((qa.verification_status=="AUTO_PASS").sum()),
    "manual_review_rows_n":int((qa.verification_status=="MANUAL_REVIEW").sum()),
    "invalid_source_field_rows_n":int((qa.verification_status=="INVALID_SOURCE_FIELD").sum()),
    "eligible_manual_review_rows_n":int(len(eligible_nonpass)),
    "eligible_manual_review_unique_urls_n":int(eligible_nonpass.source_url.nunique()),
    "rules":{
        "membership_changed":False,
        "event_rows_deleted":False,
        "event_labels_changed":False,
        "pairability_used":False,
        "chronology_used":False,
        "astrology_used":False,
        "control_used":False,
        "confirm_researched":False
    },
    "next_rule":(
        "If eligible_manual_review_rows_n > 0, send V5_SOURCE_VERIFICATION_ELIGIBLE_MANUAL_REVIEW.csv "
        "for independent web verification. Do not freeze events yet. "
        "If zero, proceed to final event QA/freeze notebook."
    )
}
summary_path=OUT/"V5_SOURCE_VERIFICATION_GATE_SUMMARY.json"
json.dump(summary,open(summary_path,"w",encoding="utf-8"),ensure_ascii=False,indent=2)

decision={
    "version":"V5_SOURCE_VERIFICATION_GATE_DECISION_V1",
    "status":status,
    "assembled_events_sha256":sha256_file(assembled_path),
    "worklist_sha256":sha256_file(work_path),
    "auto_results_sha256":sha256_file(qa_path),
    "eligible_manual_review_sha256":sha256_file(eligible_manual_path),
    "summary_sha256":sha256_file(summary_path),
    "event_freeze_allowed":len(eligible_nonpass)==0,
    "pairing_allowed":False,
    "astrology_allowed":False
}
decision_path=OUT/"V5_SOURCE_VERIFICATION_GATE_DECISION.json"
json.dump(decision,open(decision_path,"w",encoding="utf-8"),ensure_ascii=False,indent=2)

print(json.dumps(summary,ensure_ascii=False,indent=2))


{
  "version": "V5_SOURCE_VERIFICATION_GATE_SUMMARY_V1",
  "created_at": "2026-08-17T04:36:44",
  "status": "V5_SOURCE_VERIFICATION_MANUAL_BACKFILL_REQUIRED",
  "subjects_n": 160,
  "event_rows_n": 481,
  "eligible_rows_n": 405,
  "excluded_rows_n": 76,
  "unique_source_urls_n": 313,
  "auto_pass_rows_n": 348,
  "manual_review_rows_n": 133,
  "invalid_source_field_rows_n": 0,
  "eligible_manual_review_rows_n": 118,
  "eligible_manual_review_unique_urls_n": 77,
  "rules": {
    "membership_changed": false,
    "event_rows_deleted": false,
    "event_labels_changed": false,
    "pairability_used": false,
    "chronology_used": false,
    "astrology_used": false,
    "control_used": false,
    "confirm_researched": false
  },
  "next_rule": "If eligible_manual_review_rows_n > 0, send V5_SOURCE_VERIFICATION_ELIGIBLE_MANUAL_REVIEW.csv for independent web verification. Do not freeze events yet. If zero, proceed to final event QA/freeze notebook."
}


## Return to ChatGPT

After `Kernel Restart -> Run All`, send:

```text
V5_SOURCE_VERIFICATION_GATE_DECISION.json
V5_SOURCE_VERIFICATION_GATE_SUMMARY.json
V5_SOURCE_VERIFICATION_ELIGIBLE_MANUAL_REVIEW.csv
```

If the eligible manual-review CSV has zero rows, the final event-freeze notebook can be built immediately.

If it has rows, **do not edit or delete them**. Send the file back so those rows can be independently web-verified and an immutable verification override file can be produced.

Do not pair events or generate astrology yet.
